# Merge Severity Labels with Teacher Labels

**VERSION: merge-severity-v1** (2026-10-08)

Merges the 9 Antigravity severity batch JSONs (4,276 reports, 100% gold-validated)
into the stevenleehans teacher label table.

Replaces teacher's 4 weakest findings (Synovitis, PF OA, Medial OA, Lateral OA)
with binary targets from severity grades. Keeps the other 8 from teacher.

**Attach these datasets:**
1. Competition data (rsna-knee-abnormality-detection)
2. stevenleehans/rsna-knee-llm-report-labels (teacher labels)
3. Your severity-batches-v1 dataset (9 JSON files)

**Output:** `/kaggle/working/merged_labels_v1.csv` — upload this as a new dataset for training.

In [ ]:
import glob, json, os
import pandas as pd
import numpy as np

print("merge-severity-v1")

# Find batch JSONs
batch_files = sorted(glob.glob("/kaggle/input/severity-batches-v1/*.json"))
print(f"Found {len(batch_files)} batch files")
assert len(batch_files) == 9, f"Expected 9, got {len(batch_files)}"
for f in batch_files:
    print(f"  {os.path.basename(f)}")

# Find teacher CSV
teacher_path = "/kaggle/input/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv"
assert os.path.exists(teacher_path), f"Teacher CSV not found: {teacher_path}"
print(f"\nTeacher CSV: {teacher_path}")

In [ ]:
# Load severity batches
sev = {}
for fp in batch_files:
    with open(fp) as f:
        data = json.load(f)
    if isinstance(data, dict):
        data = list(data.values())
    for d in data:
        uid = d["StudyInstanceUID"]
        assert uid not in sev, f"Duplicate UID: {uid}"
        sev[uid] = d
print(f"Loaded {len(sev)} severity grades")
assert len(sev) == 4276, f"Expected 4276, got {len(sev)}"

# Load teacher
teacher = pd.read_csv(teacher_path)
print(f"Teacher: {teacher.shape}")
print(f"Columns: {list(teacher.columns)}")

In [ ]:
# Severity -> binary mapping
SEV_TO_BIN = {"none": 0, "mild": 1, "moderate": 1, "severe": 1}

# Map severity keys to teacher column names
SEV_COLS = {
    "synovitis": "Synovitis",
    "pfoa": "PF OA",
    "medial_oa": "Medial OA",
    "lateral_oa": "Lateral OA",
}

merged = teacher.copy()
replaced = 0
missing = 0

for _, row in teacher.iterrows():
    uid = str(row["StudyInstanceUID"])
    if uid not in sev:
        missing += 1
        continue
    s = sev[uid]
    idx = teacher[teacher["StudyInstanceUID"] == row["StudyInstanceUID"]].index[0]
    for sev_key, col in SEV_COLS.items():
        grade = str(s.get(sev_key, "none")).lower().strip()
        assert grade in SEV_TO_BIN, f"Invalid grade '{grade}' for {uid}"
        merged.at[idx, col] = SEV_TO_BIN[grade]
        replaced += 1

print(f"Replaced {replaced} cells across 4 findings")
print(f"UIDs missing severity (kept teacher): {missing}")

# Verify no NaNs
assert not merged.isnull().any().any(), "NaNs in merged output!"
print("No NaNs - OK")

In [ ]:
out_path = "/kaggle/working/merged_labels_v1.csv"
merged.to_csv(out_path, index=False)
print(f"Saved to {out_path}")
print(f"Shape: {merged.shape}")

# Quick stats
print("\nPer-finding positive rates (merged):")
for col in ["Synovitis", "PF OA", "Medial OA", "Lateral OA"]:
    rate = (merged[col] > 0.5).mean()
    print(f"  {col}: {rate:.3f}")

print("\nDone! Upload /kaggle/working/merged_labels_v1.csv as a new Kaggle dataset.")